# Chapter 1

This Jupyter Notebook implements key parts of Chapter 1 of the book "An Introduction to Tsetlin Machines". The book can be downloaded here https://tsetlinmachine.org.

## Boolean Features

You first need observations that the Tsetlin machine can learn from. The following dataset contains three objects of type _Car_. They are characterized by five Boolean features: _Four Wheels_, _Transports People_, _Wings_, _Yellow_, and _Blue_. The first object in the dataset, for instance, has _Four Wheels_, _Transports People_, does not have _Wings_, is _Blue_, but not _Yellow_. 

In [1239]:
cars = [
    {'Four Wheels':True, 'Transports People':True, 'Wings':False, 'Yellow':False, 'Blue':True},
    {'Four Wheels':True, 'Transports People':True, 'Wings':False, 'Yellow':True, 'Blue':False},
    {'Four Wheels':True, 'Transports People':True, 'Wings':False, 'Yellow':True, 'Blue':False}
]

## Creating Patterns with AND and NOT

The Tsetlin machine learns to recognize different object types by creating _if-then rules_. Each rule has the form: __if__ _condition_ __then__ _class_.

The _condition_ is an __and__-expression while the _class_ is a type of object. You can use the method `evaluate_condition()` to evaluate the condition of a rule:

In [1240]:
def evaluate_condition(observation, condition):
    truth_value_of_condition = True
    for feature in observation:
        if feature in condition and observation[feature] == False:
            truth_value_of_condition = False
            break
        if 'NOT ' + feature in condition and observation[feature] == True:
            truth_value_of_condition = False
            break
    return truth_value_of_condition

For instance, create the condition

In [1241]:
example_condition = ['Four Wheels', 'Transports People', 'NOT Wings']

This condition means: _Four Wheels_ __and__ _Transports People_ __and__ __not__ _Wings_. Now, evaluate the condition on the first object in the dataset:

In [1242]:
evaluate_condition(cars[0], example_condition)

True

Since the first object in the dataset has _Four Wheels_, _Transports People_, and does not have _Wings_, the __and__-expression becomes _True_.

## Learning Frequent Patterns with Type I Feedback

### Rule Memory

You are now about to discover how a single if-then rule learns to recognizing frequent _Car_ patterns. To this end, the rule has its own memory for storing literals. The memory simulates memorization and forgetting, visualized in the figure below,

<div>
<img src="attachment:Tsetlin_Machine_Memory_1_1.png" width="800"/>
</div>

The y-axis ranges from 1 to 10 and measures how deeply the memory stores each literal:

* Values 1 to 5 stand for _Forgotten_. Value 1 means maximally forgotten while value 5 means almost memorized. Literals in this range _do not_ take part in the rule's condition.
* Values 6 to 10 mean _Memorized_. Value 6 stands for lightly retained, and value 10 represents maximally memorized. Literals in this range take part in the condition of the rule.

You can implement the memory as a class:

In [1243]:
class Memory:
    def __init__(self, forget_value, memorize_value, memory):
        self.memory = memory
        self.forget_value = forget_value
        self.memorize_value = memorize_value
    
    def get_memory(self):
        return self.memory
    
    def get_literals(self):
        return list(self.memory.keys())
    
    def get_condition(self):
        condition = []
        for literal in self.memory:
            if self.memory[literal] >= 6:
                condition.append(literal)
        return condition
        
    def memorize(self, literal):
        if random() <= self.memorize_value and self.memory[literal] < 10:
            self.memory[literal] += 1
            
    def forget(self, literal):
        if random() <= self.forget_value and self.memory[literal] > 1:
            self.memory[literal] -= 1
            
    def memorize_always(self, literal):
        if  self.memory[literal] < 10:
            self.memory[literal] += 1

To create the memory for the _Car_ if-then rule, you first set `forget_value`, which decides how quickly the memory forgets. You also set `memorize_value`, determining memorization speed. Finally, you add the literals and their starting memory positions.

In [1244]:
car_rule = Memory(0.9, 0.1, {'Four Wheels':10, 'NOT Four Wheels':2, 'Transports People':9, 'NOT Transports People':3, 'Wings':1, 'NOT Wings':5, 'Yellow':1, 'NOT Yellow':4, 'Blue':6, 'NOT Blue':4})

The `get_condition()` method extracts the literals in memory position 6 to 10. You can try `get_condition()` on the above example memory.

In [1245]:
car_rule.get_condition()

['Four Wheels', 'Transports People', 'Blue']

The condition is _Four Wheels_ __and__ _Transports People_ __and__ _Blue_. You then get the _Car_ rule:

In [1246]:
print("IF " + " AND ".join(car_rule.get_condition()) + " THEN Car")

IF Four Wheels AND Transports People AND Blue THEN Car


### Memorization and Forgetting

You memorize a literal by incrementing its position in memory unless already _Maximally Memorized_ in position 10. The memorization thus pushes the literals deeper into the rule's memory. The `memorize()` method implements memorization.

You forget a literal by decrementing its position in memory unless already Maximally Forgotten in position 1. The `forget()` method implements forgetting.

Both `memorize()` and `forget()` are randomized, hence we need the `random` method from the `random` module:

In [1247]:
from random import random

Try out forget two times on literal _Blue_ and observe how the memory updates:

In [1248]:
car_rule.forget('Blue')
car_rule.forget('Blue')
print(car_rule.memory)

{'Four Wheels': 10, 'NOT Four Wheels': 2, 'Transports People': 9, 'NOT Transports People': 3, 'Wings': 1, 'NOT Wings': 5, 'Yellow': 1, 'NOT Yellow': 4, 'Blue': 4, 'NOT Blue': 4}


_Blue_ is now either in position 4, 5, or 6 depending on how many times `random()` was equal to or below 0.9.

### Initialization

When learning begins the rule starts up with all the literals in memory position 5. That is, the literals are about to be _Memorized_ but currently _Forgotten_.

In [1249]:
car_rule = Memory(0.9, 0.1, {'Four Wheels':5, 'NOT Four Wheels':5, 'Transports People':5, 'NOT Transports People':5, 'Wings':5, 'NOT Wings':5, 'Yellow':5, 'NOT Yellow':5, 'Blue':5, 'NOT Blue':5})

### Type I Feedback

The `type_i_feedback()` method produces frequent patterns with two learning steps:

* Check if the condition part of the rule is _True_ by assessing the object's literals. If the condition part is _True_, then memorize all the literals that are _True_ for the object.

* Forget all remaining literals.

In [1250]:
def type_i_feedback(observation, memory):
    remaining_literals = memory.get_literals()
    if evaluate_condition(observation, memory.get_condition()) == True:
        for feature in observation:
            if observation[feature] == True:
                memory.memorize(feature)
                remaining_literals.remove(feature)
            elif observation[feature] == False:
                memory.memorize('NOT ' + feature)
                remaining_literals.remove('NOT ' + feature)
    for literal in remaining_literals:
        memory.forget(literal)

### Example Run

The following code randomly selects a car and then updates the memory. This procedure is repeated 100 times.

In [1251]:
from random import choice

for i in range(100):
    observation_id = choice([0,1,2])
    type_i_feedback(cars[observation_id], car_rule)

The memory now looks like this:

In [1252]:
print(car_rule.get_memory())

{'Four Wheels': 10, 'NOT Four Wheels': 1, 'Transports People': 10, 'NOT Transports People': 1, 'Wings': 1, 'NOT Wings': 10, 'Yellow': 1, 'NOT Yellow': 1, 'Blue': 1, 'NOT Blue': 1}


Observe that some of the literals are now either deeply memorized or forgotten. You can print out the resulting rule:

In [1253]:
print("IF " + " AND ".join(car_rule.get_condition()) + " THEN Car")

IF Four Wheels AND Transports People AND NOT Wings THEN Car


Congratulations, you have now learnt a frequent _Car_ pattern.

## Increasing Discrimination Power with Type II Feedback

You can use a dataset with planes to ensure that the _Car_ rule is discriminative:

In [1254]:
planes = [
    {'Four Wheels':True, 'Transports People':True, 'Wings':True, 'Yellow':False, 'Blue':True},
    {'Four Wheels':True, 'Transports People':False, 'Wings':True, 'Yellow':True, 'Blue':False},
    {'Four Wheels':False, 'Transports People':True, 'Wings':True, 'Yellow':False, 'Blue':True}
]

### Type II Feedback

The `type_ii_feedback()` method implements the third and final learning step:

* Check if the condition part of the rule is _True_ by assessing the object’s literals. If the condition part  is _True_, then  memorize  all _Forgotten_ literals that are _False_ for the object. This time there is no randomization – the increment is always performed.

In [1255]:
def type_ii_feedback(observation, memory):
    if evaluate_condition(observation, memory.get_condition()) == True:
        for feature in observation:
            if observation[feature] == False:
                memory.memorize_always(feature)
            elif observation[feature] == True:
                memory.memorize_always('NOT ' + feature)

### Example Run

Let us assume that you start out with the less discriminative rule:

In [1256]:
car_rule = Memory(0.9, 0.1, {'Four Wheels': 10, 'NOT Four Wheels': 1, 'Transports People': 10, 'NOT Transports People': 1, 'Wings': 1, 'NOT Wings': 1, 'Yellow': 1, 'NOT Yellow': 1, 'Blue': 1, 'NOT Blue': 1})
print("IF " + " AND ".join(car_rule.get_condition()) + " THEN Car")

IF Four Wheels AND Transports People THEN Car


The rule correctly matches all of the cars in the _Car_ dataset. However, it also erroneously matches the first object in the _Plane_ dataset:

In [1257]:
print(evaluate_condition(planes[0], car_rule.get_condition()))

True


A few rounds of Type II Feedback fix this error:

In [1258]:
for i in range(100):
    observation_id = choice([0,1,2])
    type_ii_feedback(planes[observation_id], car_rule)
print(car_rule.get_memory())

{'Four Wheels': 10, 'NOT Four Wheels': 6, 'Transports People': 10, 'NOT Transports People': 6, 'Wings': 1, 'NOT Wings': 6, 'Yellow': 6, 'NOT Yellow': 1, 'Blue': 1, 'NOT Blue': 6}


Observe how the feedback has pushed several _False_ literals to memory position 6. The updated rule becomes:

In [1259]:
print("IF " + " AND ".join(car_rule.get_condition()) + " THEN Car")

IF Four Wheels AND NOT Four Wheels AND Transports People AND NOT Transports People AND NOT Wings AND Yellow AND NOT Blue THEN Car


The rule no longer recognizes the offending plane:

In [1260]:
print(evaluate_condition(planes[0], car_rule.get_condition()))

False


Unfortunately, the new rule does not match _Car_ frequently. To make the rule both discriminative and frequent you mix Type I and Type II Feedback:

In [1261]:
for i in range(100):
    observation_id = choice([0,1,2])
    car = choice([0,1])
    if car == 1:
        type_i_feedback(cars[observation_id], car_rule)
    else:
        type_ii_feedback(planes[observation_id], car_rule)

The final rule you get is both frequent for _Car_ and distinguishes between _Car_ and _Plane_. 

In [1262]:
print("IF " + " AND ".join(car_rule.get_condition()) + " THEN Car")

IF Four Wheels AND Transports People AND NOT Wings THEN Car


The rule is also deeply memorized with all the important literals in or close to memory position 10.

In [1263]:
print(car_rule.get_memory())

{'Four Wheels': 9, 'NOT Four Wheels': 1, 'Transports People': 8, 'NOT Transports People': 1, 'Wings': 1, 'NOT Wings': 9, 'Yellow': 1, 'NOT Yellow': 1, 'Blue': 1, 'NOT Blue': 2}


## Putting the Pieces Together – How Several Rules Coordinate

The Tsetlin machine classifies input by voting, implemented by `classify()`. To classify, you simply count how many rules vote for each class. The Tsetlin machine then outputs the class with the most votes.`

In [1264]:
def classify(observation, recurrence_rules, non_recurrence_rules):
    vote_sum = get_vote_sum(observation, recurrence_rules, non_recurrence_rules)
    if vote_sum >= 0:
        return "Recurrence"
    else:
        return "Non-Recurrence"

In [1265]:
# Modified classify function to return votes sum instead of just the final classification

def get_vote_sum(observation, recurrence_rules, non_recurrence_rules):
    vote_sum = 0
    for rule in recurrence_rules:
        if evaluate_condition(observation, rule.get_condition()) == True:
            vote_sum += 1
    for rule in non_recurrence_rules:
        if evaluate_condition(observation, rule.get_condition()) == True:
            vote_sum -= 1
    return vote_sum
    

## Assignment 2

In [1266]:
# Part 2: Create a dataset

FEATURES = {
    'Menopause': ['ge40', 'lt40', 'premeno'],
    'Inv-nodes': ['0-2', '3-5', '6-8'],
    'Deg-malig': ['1', '2', '3']
}
 
def patient(menopause, inv_nodes, deg_malig):
    active_features = {"Menopause " + menopause, "Inv-nodes " + inv_nodes, "Deg-malig " + deg_malig}
    result = {}
    for feature in FEATURES:
        for value in FEATURES[feature]:
            feature_name = feature + " " + value
            result[feature_name] = feature_name in active_features
    return result

patients = {
    'recurrence_patients': [(1,patient('ge40', '3-5', '3')), (3,patient('ge40', '6-8', '3')), (5,patient('premeno', '0-2', '3'))],
    'non_recurrence_patients': [(2,patient('lt40', '0-2', '3')), (4,patient('ge40', '0-2', '2')), (6,patient('premeno', '0-2', '1'))]
}


# add all 18 literals, including (NOT) to a list, so that we can use it to create a memory for the rules
# Part 4 want us to classify all 6 patients with the rules from Part 3, so we need to create a memory for the rules first.
 
literals = []
literal_count = 0

for feature in FEATURES:
    for value in FEATURES[feature]:
        literals.append(feature + " " + value)
        literals.append("NOT " + feature + " " + value)
        literal_count += 2


#Check that the literals and literal_count are correct
print(literals)
print(literal_count)

print(patients['recurrence_patients'][0])

['Menopause ge40', 'NOT Menopause ge40', 'Menopause lt40', 'NOT Menopause lt40', 'Menopause premeno', 'NOT Menopause premeno', 'Inv-nodes 0-2', 'NOT Inv-nodes 0-2', 'Inv-nodes 3-5', 'NOT Inv-nodes 3-5', 'Inv-nodes 6-8', 'NOT Inv-nodes 6-8', 'Deg-malig 1', 'NOT Deg-malig 1', 'Deg-malig 2', 'NOT Deg-malig 2', 'Deg-malig 3', 'NOT Deg-malig 3']
18
(1, {'Menopause ge40': True, 'Menopause lt40': False, 'Menopause premeno': False, 'Inv-nodes 0-2': False, 'Inv-nodes 3-5': True, 'Inv-nodes 6-8': False, 'Deg-malig 1': False, 'Deg-malig 2': False, 'Deg-malig 3': True})


In [1267]:
# Part 3: the three rules from the lecture: R1 and R2 predict Recurrence, R3 predicts Non-recurrence.

conditions = {
    'R1': ['Deg-malig 3', 'NOT Menopause lt40'],
    'R2': ['Deg-malig 3'],
    'R3': ['Inv-nodes 0-2']
}

# Create a memory for the rules, with the literals and literal_count from Part 2. We will use this memory to classify the patients in Part 4. 
def create_rule_memory(condition, literals):
    rule_memory = {}
    for literal in literals:
        if literal in condition:
            rule_memory[literal] = 10
        else:
            rule_memory[literal] = 1
            # Default arguments for forget_value and memorize_value are 0.8 and 0.2, respectively. 
            # Default arguments can be adjusted as needed, can be owerwritten when calling the function.
    return Memory(0.8, 0.2, rule_memory)

R1 = create_rule_memory(conditions['R1'], literals)
R2 = create_rule_memory(conditions['R2'], literals)
R3 = create_rule_memory(conditions['R3'], literals)

recurrence_rule = [R1, R2]
non_recurrence_rule = [R3]

print(R1.get_memory())

{'Menopause ge40': 1, 'NOT Menopause ge40': 1, 'Menopause lt40': 1, 'NOT Menopause lt40': 10, 'Menopause premeno': 1, 'NOT Menopause premeno': 1, 'Inv-nodes 0-2': 1, 'NOT Inv-nodes 0-2': 1, 'Inv-nodes 3-5': 1, 'NOT Inv-nodes 3-5': 1, 'Inv-nodes 6-8': 1, 'NOT Inv-nodes 6-8': 1, 'Deg-malig 1': 1, 'NOT Deg-malig 1': 1, 'Deg-malig 2': 1, 'NOT Deg-malig 2': 1, 'Deg-malig 3': 10, 'NOT Deg-malig 3': 1}


In [1268]:
# Part 4: Classify all 6 pattients with the rules from Part 3

def classify_patients(patients, recurrence_rule, non_recurrence_rule):
    patient_classifications = {}
    for patient_type in patients:
        for number,patient in patients[patient_type]:
            # Uses the classify function Example above, slightly modified.
            patient_classifications[number] = classify(patient, recurrence_rule, non_recurrence_rule)
    return patient_classifications

def get_sum_votes_patients(patients, recurrence_rule, non_recurrence_rule):
    patient_classifications = {}
    for patient_type in patients:
        for number,patient in patients[patient_type]:
            patient_classifications[number] = get_vote_sum(patient, recurrence_rule, non_recurrence_rule)
            if patient_classifications[number] >= 0:
                patient_classifications[number] = "+" + str(patient_classifications[number])
            else:
                pass
    return patient_classifications

classifications = classify_patients(patients, recurrence_rule, non_recurrence_rule)
sum_votes = get_sum_votes_patients(patients, recurrence_rule, non_recurrence_rule)

print(classifications)
print(sum_votes)


{1: 'Recurrence', 3: 'Recurrence', 5: 'Recurrence', 2: 'Recurrence', 4: 'Non-Recurrence', 6: 'Non-Recurrence'}
{1: '+2', 3: '+2', 5: '+1', 2: '+0', 4: -1, 6: -1}


In [1269]:
# Part 5: Mix Type I and Type II Feedback with forget value 0.8 and memorize value 0.2 to learn a rule for Recurrence

from random import seed

seed(42)

def create_untrained_rule(literals, forget_value, memorize_value):
    rule_memory = {}
    for literal in literals:
        rule_memory[literal] = 5
    return Memory(forget_value, memorize_value, rule_memory)

def learn_rule(patients, recurrence, forget_value=0.8, memorize_value=0.2, iterations=500):
    rule = create_untrained_rule(literals, forget_value, memorize_value)

    # If recurrence is True, we will use Type I feedback for recurrence patients and Type II feedback for non-recurrence patients.
    # Meaning we want to learn a rule for recurrence patients. 
    if recurrence:
        for i in range(iterations):
            coin_flip = choice([0, 1])
            if coin_flip == 1:
                # Pick a random patient and discard the ID, keeping only the patient data
                _, random_recurrence_patient = choice(patients['recurrence_patients'])
                type_i_feedback(random_recurrence_patient, rule)
            else:
                _, random_non_recurrence_patient = choice(patients['non_recurrence_patients'])
                type_ii_feedback(random_non_recurrence_patient, rule)
                
    # If recurrence is False, we will use Type I feedback for non-recurrence patients and Type II feedback for recurrence patients.
    # Meaning we want to learn a rule for non-recurrence patients.
    else:
        for i in range(iterations):
            coin_flip = choice([0, 1])
            if coin_flip == 1:
                _, random_non_recurrence_patient = choice(patients['non_recurrence_patients'])
                type_i_feedback(random_non_recurrence_patient, rule)
            else:
                _, random_recurrence_patient = choice(patients['recurrence_patients'])
                type_ii_feedback(random_recurrence_patient, rule)
    return rule


learned_recurrence_rule = learn_rule(patients, recurrence=True, forget_value=0.8, memorize_value=0.2, iterations=500)

print ("IF " + " AND ".join(learned_recurrence_rule.get_condition()) + " THEN Recurrence")

IF NOT Menopause lt40 AND NOT Deg-malig 1 AND NOT Deg-malig 2 AND Deg-malig 3 THEN Recurrence


In [1270]:
# Part 6: Mix Type I and Type II Feedback with forget value 0.8 and memorize value 0.2 to learn a rule for Non-recurrence

# Is not needed to pass forget_value and memorize_value as arguments, since they have default values of 0.8 and 0.2, respectively in the function definition, the same also applies to the recurrence parameter.
learned_non_recurrence_rule = learn_rule(patients, recurrence=False, forget_value=0.8, memorize_value=0.2, iterations=500)

print ("IF " + " AND ".join(learned_non_recurrence_rule.get_condition()) + " THEN Non-Recurrence")

IF Inv-nodes 0-2 AND NOT Inv-nodes 3-5 AND NOT Inv-nodes 6-8 THEN Non-Recurrence


In [1271]:
# Part 7

learned_recurrence_rule_05_05 = learn_rule(patients, recurrence=True, forget_value=0.5, memorize_value=0.5, iterations=500)
learned_non_recurrence_rule_05_05 = learn_rule(patients, recurrence=False, forget_value=0.5, memorize_value=0.5, iterations=500)

print ("IF " + " AND ".join(learned_non_recurrence_rule_05_05.get_condition()) + " THEN Non-Recurrence")
print ("IF " + " AND ".join(learned_recurrence_rule_05_05.get_condition()) + " THEN Recurrence")

IF Menopause lt40 AND NOT Inv-nodes 6-8 THEN Non-Recurrence
IF Menopause ge40 AND NOT Menopause lt40 AND NOT Menopause premeno AND NOT Inv-nodes 0-2 AND NOT Deg-malig 1 AND NOT Deg-malig 2 AND Deg-malig 3 THEN Recurrence


In [1272]:
# Part 8

learned_recurrence_rule_02_08 = learn_rule(patients, recurrence=True, forget_value=0.2, memorize_value=0.8, iterations=500)
learned_non_recurrence_rule_02_08 = learn_rule(patients, recurrence=False, forget_value=0.2, memorize_value=0.8, iterations=500)

print ("IF " + " AND ".join(learned_non_recurrence_rule_02_08.get_condition()) + " THEN Non-Recurrence")
print ("IF " + " AND ".join(learned_recurrence_rule_02_08.get_condition()) + " THEN Recurrence")

IF Menopause ge40 AND NOT Menopause lt40 AND NOT Menopause premeno AND Inv-nodes 0-2 AND NOT Inv-nodes 3-5 AND NOT Inv-nodes 6-8 AND NOT Deg-malig 1 AND Deg-malig 2 AND NOT Deg-malig 3 THEN Non-Recurrence
IF Menopause ge40 AND NOT Menopause lt40 AND NOT Menopause premeno AND NOT Inv-nodes 0-2 AND Inv-nodes 3-5 AND NOT Inv-nodes 6-8 AND NOT Deg-malig 1 AND NOT Deg-malig 2 AND Deg-malig 3 THEN Recurrence


In [1273]:
# Get average rule length for the learned rules

forget_memorize_tuples = [(0.8, 0.2), (0.5, 0.5), (0.2, 0.8)]

def average_rule_length(forget_value, memorize_value, runs=25):
    total_recurrence_length = 0
    total_non_recurrence_length = 0
    for r in range(runs):
        learned_recurrence_rule = learn_rule(patients, recurrence=True, forget_value=forget_value, memorize_value=memorize_value, iterations=500)
        learned_non_recurrence_rule = learn_rule(patients, recurrence=False, forget_value=forget_value, memorize_value=memorize_value, iterations=500)
        #gets the length of the learned rules, which is the number of literals in the condition of the rule
        recurrence_rule_length = len(learned_recurrence_rule.get_condition())
        non_recurrence_rule_length = len(learned_non_recurrence_rule.get_condition())
        
        # adds the lengths of the learned rules to the total lengths for recurrence and non-recurrence rules
        total_recurrence_length += recurrence_rule_length
        total_non_recurrence_length += non_recurrence_rule_length
    # calculates the average lengths of the learned rules for recurrence and non-recurrence rules by dividing the total lengths by the number of runs.
    average_recurrence_length = total_recurrence_length / (runs)
    average_non_recurrence_length = total_non_recurrence_length / (runs)

    return average_recurrence_length, average_non_recurrence_length

for tuples in forget_memorize_tuples:
    forget_value, memorize_value = tuples
    average_recurrence_length, average_non_recurrence_length = average_rule_length(forget_value, memorize_value)

    print(f"Forget value: {forget_value}, Memorize value: {memorize_value}")
    print(f"Average Recurrence Rule Length: {average_recurrence_length}")
    print(f"Average Non-Recurrence Rule Length: {average_non_recurrence_length}")
    print("--------------------------------------------------")

Forget value: 0.8, Memorize value: 0.2
Average Recurrence Rule Length: 3.56
Average Non-Recurrence Rule Length: 2.36
--------------------------------------------------
Forget value: 0.5, Memorize value: 0.5
Average Recurrence Rule Length: 6.52
Average Non-Recurrence Rule Length: 4.48
--------------------------------------------------
Forget value: 0.2, Memorize value: 0.8
Average Recurrence Rule Length: 8.76
Average Non-Recurrence Rule Length: 8.76
--------------------------------------------------


## Step 1-4 — Inference by classifications of the patients with hand written rules R1, R2 & R3

| # | Recurrence | Vote sum | Correct? |
|---|---|---|---|
| 1 | yes | +2 | yes |
| 2 | no | 0 | **no** |
| 3 | yes | +2 | yes |
| 4 | no | −1 | yes |
| 5 | yes | +1 | yes |
| 6 | no | −1 | yes |

All correct except for Patient #2 that has vote sum of 0, a tie. It then gets classified as recurrence when it actually is non-recurrence

## Steps 5–8 — Learned rules

**forget 0.8 / memorize 0.2**

- Recurrence: IF `NOT Menopause lt40` AND `NOT Deg-malig 1` AND `NOT Deg-malig 2` AND `Deg-malig 3` THEN Recurrence
- Non-recurrence: IF `Inv-nodes 0-2` AND `NOT Inv-nodes 3-5` AND `NOT Inv-nodes 6-8` THEN Non-Recurrence

**forget 0.5 / memorize 0.5**

- Recurrence: `LITERAL` AND ... (N literals)
- Non-recurrence: `LITERAL` AND ... (N literals)

**forget 0.2 / memorize 0.8**

- Recurrence: N literals, including `LITERAL`
- Non-recurrence: N literals, including `LITERAL`

## Rule length (average over 25 runs)

| Forget | Memorize | Average literals (Recurrence)| Average literals (Non-recurrence) |
|---|---|---|---|
| 0.8 | 0.2 | 3.56 | 2.36 |
| 0.5 | 0.5 | 6.52 | 4.48 |
| 0.2 | 0.8 | 8.76 | 8.76 |

## What this means

**Rules get longer when the forget value goes down.**

At **0.8 / 0.2** the machine forgets quickly, so only literals that keep getting reinforced survive. The rules are short and general. The recurrence rule (≈4 literals) is essentially the lecture's R1: `Deg-malig 3` AND `NOT Menopause lt40`, plus the redundant `NOT Deg-malig 1` and `NOT Deg-malig 2`.

At **0.5 / 0.5** literals that are true for only some of the patients in the class start surviving, and the rules become longer and more specific.

At **0.2 / 0.8** the machine forgets almost nothing, and the rules end up close to 9 literals. Every patient has exactly 9 true literals (3 features + 6 `NOT` literals), so a 9-literal rule is a full description of a single patient rather than a pattern.

So a high forget value gives **generalization** and a low forget value gives **overfitting**.

**The non-recurrence rule is harder to learn.** The non-recurrence patients only have `Inv-nodes 0-2` in common, and recurrence patient #5 has it too. Type I and Type II feedback therefore pull in opposite directions, and at 0.8 / 0.2 the rule stays very short (≈1.7 literals), often covering only some of its class, e.g. `NOT Deg-malig 3`, which matches patients #4 and #6. A single AND-rule cannot capture this class, which is why a Tsetlin machine uses several rules per class and lets them vote.